In [ ]:
import cv2
import mediapipe as mp
from deepface import DeepFace
import os

# cam = cv2.VideoCapture(1)
mp_face_detection = mp.solutions.face_detection.FaceDetection()
face_detection = mp_face_detection
mp_draw = mp.solutions.drawing_utils

while True:
    # success, img=cam.read()

    img = cv2.imread("/content/drive/MyDrive/database/pic1.jpg")

    # if not success:
    #     print("Ignoring empty camera frame.")
    #     break
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_detection.process(img_rgb)

    if results.detections:
        for detection in results.detections:
            mp_draw.draw_detection(img, detection)
            ih, iw, _ = img.shape
            bboxC = detection.location_data.relative_bounding_box
            bbox = int(bboxC.xmin *iw), int(bboxC.ymin *ih), int(bboxC.width *iw), int(bboxC.height *ih)
            x, y, w, h = bbox
            face = img[y:y+h, x:x+w]
            try:
                dfs = DeepFace.find(face, db_path="database", model_name="VGG-Face", enforce_detection=False)
                if not dfs[0].empty:    #if dfs is not empty
                    identity_path = dfs[0]['identity'][0]
                    name = os.path.splitext(os.path.basename(identity_path))[0]
                    cv2.putText(img, name, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            except Exception as e:
                print("Face not found in database")
    img=cv2.flip(img, 1)
    cv2.imshow("face", img)
    if cv2.waitKey(1) & 0xff == ord("x"):
        break

# cam.release()
# cv2.destroyAllWindows()